# BP1 Gate 2 — CFPB <-> BANKING77 Data Integration & Taxonomy Mapping
**Customer360 Navigator Enterprise Suite**

## Purpose
Implements Master Execution Plan Section 6 (Integration Architecture) and Section 7 (BP1
Methodology, Gate 2 — Data Verification & Feature/Taxonomy Engineering): applies the documented
CFPB<->BANKING77 common-taxonomy crosswalk (`configs/taxonomy_mapping.yaml`,
`docs/data_dictionary/CFPB_BANKING77_TAXONOMY_MAPPING.md`) to the real raw data, produces the
mapping-coverage audit table the Master Plan requires, and writes the Customer360 Gold
common-taxonomy layer. This is shared infrastructure feeding BP1, BP2, BP4, BP6 and BP8 wherever
the Master Plan calls for CFPB<->BANKING77 integration (Table 1) — it is not itself a trained
model or a BP1 Gate-3/4 modeling notebook.

## Standing rules this notebook follows
- **Execution boundary**: Claude wrote this notebook; it does not run it. You run it on your own
  machine and the real, live-measured numbers below (including the drift check in Section 4)
  become the project's integration record.
- **Zero-fabrication**: the taxonomy mapping itself is a *disclosed judgment call*, not a
  statistical result — every bucket's confidence (HIGH/MEDIUM/LOW) and rationale is documented in
  `CFPB_BANKING77_TAXONOMY_MAPPING.md`. This is **not** a row-level join: CFPB and BANKING77 share
  no common identifier, and this CFPB export has no narrative-text column, so no text model is
  jointly trained across both datasets here.
- **WARP**: `configure_performance()` (src/utils/performance_setup.py) is called first, before any
  heavy import, per `LESSONS_LEARNED_APPLIED.md` #5. Polars lazy scan for the 322MB CFPB file — it
  is never fully materialized in memory at once. Output written as Parquet, not CSV.
- **Idempotent**: re-running overwrites `data/processed/*_common_taxonomy_gold.parquet` in place.
- **PROJECT_STRUCTURE_LOCKED.md rule #3**: project root resolved via `C360_PROJECT_ROOT` env var
  override first, then a bounded upward walk — never a hardcoded absolute path.

## Inputs
- `data/raw/cfpb_complaints.csv` (real CFPB export, 1,048,575 data rows — see
  `docs/data_dictionary/RAW_DATA_MANIFEST.md` for its documented limitations: no narrative text,
  no demographic field, and an exact-2^20 row count consistent with an Excel export cap)
- `data/external/banking77_{train,test}.csv`, `data/external/banking77_categories.json`
- `configs/taxonomy_mapping.yaml` (the reviewed crosswalk this notebook applies — it does not
  invent one)

## Outputs (idempotent overwrite-in-place)
- `data/processed/cfpb_common_taxonomy_gold.parquet`
- `data/processed/banking77_common_taxonomy_gold.parquet`
- `notebooks/bp1_customer_intent_classification/artifacts/taxonomy_mapping_coverage_report.csv`

## Prerequisites
`pip install -r requirements.txt` from the project root (needs `polars`, `pyyaml`, `psutil`).

## If a structural check below fails
It raises `AssertionError` naming the failing check, including a **live drift check**: this
notebook recomputes the real CFPB Product distribution from the actual file at run time and
compares it against the counts baked into `taxonomy_mapping.yaml` at authoring time. If the raw
file has changed since the mapping was written, this notebook FAILS rather than silently reporting
stale coverage numbers — fix the mapping config, don't edit the check.


In [ ]:

"""
Customer360 Navigator Enterprise Suite - BP1 Gate 2 data integration / taxonomy mapping notebook.
Single consolidated code cell (platform convention). Idempotent - safe to re-run.
"""

import os, sys, json, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent

    # Bounded downward search (depth <= 3, skips hidden dirs): handles the case where the
    # kernel's cwd is a PARENT of the project folder (real bug hit on 00_hardware_benchmark.ipynb
    # - see LESSONS_LEARNED_APPLIED.md - a common cause is VS Code's Jupyter extension defaulting
    # the kernel cwd to the workspace root instead of the notebook's own folder).
    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)

    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )

PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {(PROJECT_ROOT / 'PROJECT_STRUCTURE_LOCKED.md').exists()}")

# ============================================================
# SECTION 2: WARP performance configuration - FIRST, before any heavy import
# (LESSONS_LEARNED_APPLIED.md #5 - thread env vars must be set before BLAS libraries import)
# ============================================================
from utils.performance_setup import configure_performance, memory_headroom_gb

perf_summary = configure_performance(project_root=PROJECT_ROOT)
print(f"[WARP] Headroom before heavy work: {memory_headroom_gb()} GB")

# ============================================================
# SECTION 3: Heavy imports (only after WARP configuration)
# ============================================================
import polars as pl
from IPython.display import display

from taxonomy.taxonomy_mapper import (
    load_mapping_config,
    load_banking77_with_bucket,
    load_cfpb_with_bucket,
    mapping_coverage_report,
    build_common_taxonomy_layer,
)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_EXTERNAL = PROJECT_ROOT / "data" / "external"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CONFIGS_DIR = PROJECT_ROOT / "configs"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp1_customer_intent_classification" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

CFPB_PATH = DATA_RAW / "cfpb_complaints.csv"
B77_TRAIN_PATH = DATA_EXTERNAL / "banking77_train.csv"
B77_TEST_PATH = DATA_EXTERNAL / "banking77_test.csv"
B77_CATEGORIES_PATH = DATA_EXTERNAL / "banking77_categories.json"
MAPPING_CONFIG_PATH = CONFIGS_DIR / "taxonomy_mapping.yaml"

for p in (CFPB_PATH, B77_TRAIN_PATH, B77_TEST_PATH, B77_CATEGORIES_PATH, MAPPING_CONFIG_PATH):
    if not p.exists():
        raise FileNotFoundError(
            f"Required input not found: {p}. Confirm the raw-data copy step "
            "(RAW_DATA_MANIFEST.md) and the taxonomy mapping generation step both completed."
        )

# ============================================================
# SECTION 4: Load the reviewed mapping config + LIVE drift check against the real CFPB file
# Zero-fabrication: never trust taxonomy_mapping.yaml's baked-in counts without re-measuring
# ============================================================
mapping = load_mapping_config(MAPPING_CONFIG_PATH)
print(f"[OK] Loaded taxonomy_mapping.yaml: {len(mapping['banking77_category_to_bucket'])} "
      f"BANKING77 categories mapped, {len(mapping['common_taxonomy_buckets'])} common-taxonomy buckets.")

live_product_counts = (
    pl.scan_csv(CFPB_PATH, dtypes={"Product": pl.Categorical})
    .group_by("Product")
    .agg(pl.len().alias("row_count"))
    .collect()
)
live_counts_dict = dict(zip(live_product_counts["Product"].cast(pl.Utf8).to_list(),
                             live_product_counts["row_count"].to_list()))
documented_counts_dict = {d["product"]: d["row_count"] for d in mapping["cfpb_product_distribution"]}

drift = {
    product: {"documented": documented_counts_dict.get(product), "live": live_counts_dict.get(product)}
    for product in set(documented_counts_dict) | set(live_counts_dict)
    if documented_counts_dict.get(product) != live_counts_dict.get(product)
}
if drift:
    print("[DRIFT DETECTED] CFPB Product distribution has changed since taxonomy_mapping.yaml was "
          f"authored: {json.dumps(drift, indent=2)}")
else:
    print("[OK] Live CFPB Product distribution matches taxonomy_mapping.yaml exactly - no drift.")

# ============================================================
# SECTION 5: Apply the mapping to BANKING77 (eager - small file) and CFPB (lazy - 322MB file)
# ============================================================
banking77_df = load_banking77_with_bucket(B77_TRAIN_PATH, B77_TEST_PATH, B77_CATEGORIES_PATH, mapping)
print(f"[OK] BANKING77 loaded with bucket: {banking77_df.height} rows "
      f"({banking77_df.filter(pl.col('split') == 'train').height} train / "
      f"{banking77_df.filter(pl.col('split') == 'test').height} test)")

cfpb_lazy = load_cfpb_with_bucket(CFPB_PATH, mapping)

# ============================================================
# SECTION 6: Mapping coverage report (the audit table the Master Plan requires)
# ============================================================
coverage = mapping_coverage_report(cfpb_lazy, banking77_df, mapping)
coverage_pd = coverage.to_pandas()
print("\n=== TAXONOMY MAPPING COVERAGE ===")
display(coverage_pd)

out_of_scope_row = coverage.filter(pl.col("common_taxonomy_bucket") == "OUT_OF_SCOPE_NO_BANKING77_OVERLAP")
out_of_scope_fraction = (
    float(out_of_scope_row["cfpb_fraction"][0]) if out_of_scope_row.height > 0 else 0.0
)
print(f"\n[FINDING] {out_of_scope_fraction:.1%} of the CFPB file falls outside any bucket with "
      "BANKING77 overlap (dominated by credit reporting, debt collection, mortgage, and loan "
      "products BANKING77 was never built to cover). See CFPB_BANKING77_TAXONOMY_MAPPING.md "
      "section 2 for the documented figure this should match (~93.45% out-of-scope by the "
      "authored mapping, i.e. ~6.55% in-scope).")

coverage_report_path = ARTIFACTS_DIR / "taxonomy_mapping_coverage_report.csv"
coverage_pd.to_csv(coverage_report_path, index=False)
print(f"[SAVED] {coverage_report_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 7: Write the Customer360 Gold common-taxonomy layer (Parquet, WARP)
# ============================================================
gold_summary = build_common_taxonomy_layer(cfpb_lazy, banking77_df, DATA_PROCESSED)
print(f"\n[SAVED] CFPB Gold: {gold_summary['cfpb_gold_path']} ({gold_summary['cfpb_rows_written']} rows)")
print(f"[SAVED] BANKING77 Gold: {gold_summary['banking77_gold_path']} ({gold_summary['banking77_rows_written']} rows)")

# ============================================================
# SECTION 8: Structural integrity checks (never edited to force a pass)
# ============================================================
checks = [
    ("mapping_config_has_77_categories", len(mapping["banking77_category_to_bucket"]) == 77),
    # 13,083 (not 13,100) is the REAL polars-parsed row count - see LESSONS_LEARNED_APPLIED.md #14:
    # RAW_DATA_MANIFEST.md's original 13,100 figure came from a naive line count (wc -l), which
    # over-counts whenever a quoted `text` field contains an embedded newline; a real CSV parser
    # (Polars, used here) correctly treats each such field as one logical row, not several.
    ("banking77_loaded_all_rows", banking77_df.height == 13_083),
    ("banking77_no_unmapped_category", (banking77_df["common_taxonomy_bucket"] == "UNMAPPED_UNKNOWN_CATEGORY").sum() == 0),
    ("no_live_drift_vs_documented_mapping", len(drift) == 0),
    ("coverage_report_written", coverage_report_path.exists()),
    ("cfpb_gold_row_count_matches_source", gold_summary["cfpb_rows_written"] == 1_048_575),
    ("banking77_gold_row_count_matches_source", gold_summary["banking77_rows_written"] == 13_083),
]

print("\n=== INTEGRITY CHECKS ===")
all_passed = True
for name, passed in checks:
    status = "PASS" if passed else "FAIL"
    all_passed = all_passed and passed
    print(f"[{status}] {name}")

if not all_passed:
    raise AssertionError("One or more structural integrity checks FAILED - see output above. "
                          "A DRIFT or row-count mismatch means the raw data changed since this "
                          "was authored - update taxonomy_mapping.yaml and RAW_DATA_MANIFEST.md, "
                          "do not edit this check to force a pass.")

print("\n[ALL CHECKS PASSED] Data integration / taxonomy mapping complete. "
      "BP1 Gate 2 can proceed to feature engineering on top of the Gold common-taxonomy layer.")
